# Phase 4 - Dashboard export

Builds the files behind the HTML dashboard (`dashboard/index.html`). Exported as a small
**star schema** rather than one flat table, for one specific reason:

> `capacity_utilization` divides by `SUM(boat_capacity)` at **trip** grain. If
> `boat_capacity` rides on the booking rows, summing it multiplies each boat's
> capacity by the number of seats sold on it and the KPI silently reads far too
> low. Capacity therefore lives in `fact_trips`, at one row per departure.

Grains: `fact_bookings` = 1 booking · `fact_trips` = 1 departure ·
`fact_inquiries` = 1 inquiry · `dim_date` = 1 calendar day · `dim_customers` =
1 customer.

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CLEAN = os.path.join(ROOT, "data", "clean")
DASH = os.path.join(ROOT, "dashboard")
os.makedirs(DASH, exist_ok=True)

con = sqlite3.connect(os.path.join(CLEAN, "dive_ops_clean.db"))
bk = pd.read_sql("SELECT * FROM bookings", con, parse_dates=["booking_date", "dive_date"])
cu = pd.read_sql("SELECT * FROM customers", con, parse_dates=["signup_date"])
iq = pd.read_sql("SELECT * FROM inquiries", con, parse_dates=["inquiry_date"])
tp = pd.read_sql("SELECT * FROM trips", con)
wx = pd.read_sql("SELECT * FROM weather", con, parse_dates=["weather_date"])
con.close()

risk = pd.read_csv(os.path.join(CLEAN, "cancellation_risk_scores.csv"),
                   parse_dates=["dive_date"])[["booking_id", "risk", "decile"]]
kpis = pd.read_csv(os.path.join(CLEAN, "kpi_summary.csv"))
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

## `dim_date`

One row per calendar day in the window, carrying both season notions and the
sea condition. This is the table the season slicer binds to.

In [2]:
dim_date = wx.rename(columns={"weather_date": "date"}).copy()
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["month"].map(lambda m: MONTHS[m - 1])
dim_date["year_month"] = dim_date["date"].dt.to_period("M").astype(str)
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = (dim_date["date"].dt.weekday >= 5).astype(int)
# A-OQ4 trading seasons (docs/BRD.md section 6): Peak Dec-Mar,
# Shoulder Apr-Jun + Sep-Nov, Monsoon-Closed (off-peak) Jul-Aug
dim_date["season"] = np.select(
    [dim_date["month"].isin([12, 1, 2, 3]), dim_date["month"].isin([4, 5, 6, 9, 10, 11])],
    ["Peak", "Shoulder"], default="Monsoon-Closed")
dim_date["season_year"] = np.select(
    [dim_date["date"] <= "2024-05-31", dim_date["date"] <= "2025-05-31"],
    ["2023-24 (half)", "2024-25"], default="2025-26")
dim_date["is_partial_season"] = (dim_date["season_year"] == "2023-24 (half)").astype(int)
dim_date["date"] = dim_date["date"].dt.strftime("%Y-%m-%d")
print(dim_date.head(3).to_string(index=False))
print("dim_date: %d rows" % len(dim_date))

      date sea_condition  year  month month_name year_month day_of_week  is_weekend season    season_year  is_partial_season
2024-01-01      Moderate  2024      1        Jan    2024-01      Monday           0   Peak 2023-24 (half)                  1
2024-01-02      Moderate  2024      1        Jan    2024-01     Tuesday           0   Peak 2023-24 (half)                  1
2024-01-03      Moderate  2024      1        Jan    2024-01   Wednesday           0   Peak 2023-24 (half)                  1
dim_date: 882 rows


## `fact_bookings`

Booking grain. `revenue_inr` is already null on non-completed and quarantined
rows, so `SUM(revenue_inr)` is safe to use directly for every revenue measure.
`is_completed` / `is_cancelled` / `is_no_show` are pre-computed as 0/1 so the
rate measures are plain averages and cannot be mis-specified in the dashboard.

In [3]:
fact_bookings = (bk.merge(tp[["trip_id", "dive_site", "difficulty", "boat_capacity"]],
                          on="trip_id", how="left")
                   .merge(cu[["customer_id", "nationality", "cert_level",
                              "acquisition_channel"]], on="customer_id", how="left")
                   .merge(wx.rename(columns={"weather_date": "dive_date"}),
                          on="dive_date", how="left")
                   .merge(risk, on="booking_id", how="left"))

fact_bookings["is_completed"] = (fact_bookings["status"] == "Completed").astype(int)
fact_bookings["is_cancelled"] = (fact_bookings["status"] == "Cancelled").astype(int)
fact_bookings["is_no_show"] = (fact_bookings["status"] == "No-show").astype(int)
fact_bookings["completed_dives"] = np.where(fact_bookings["is_completed"] == 1,
                                            fact_bookings["num_dives"], 0)
fact_bookings = fact_bookings.rename(columns={"risk": "cancellation_risk",
                                              "decile": "risk_decile"})
fact_bookings["cancellation_reason"] = fact_bookings["cancellation_reason"].fillna("n/a")

COLS = ["booking_id", "customer_id", "trip_id", "booking_date", "dive_date",
        "season_year", "dive_month", "dive_ym", "is_weekend", "dive_type", "num_dives",
        "completed_dives", "price_inr", "revenue_inr", "price_quarantined", "status",
        "is_completed", "is_cancelled", "is_no_show", "cancellation_reason",
        "lead_time_days", "boat_slot", "dive_site", "difficulty", "boat_capacity",
        "sea_condition", "nationality", "cert_level", "acquisition_channel",
        "cancellation_risk", "risk_decile"]
fact_bookings = fact_bookings[COLS]
for c in ["booking_date", "dive_date"]:
    fact_bookings[c] = fact_bookings[c].dt.strftime("%Y-%m-%d")

print("fact_bookings: %d rows x %d cols" % fact_bookings.shape)
print("  revenue check: Rs %.0f" % fact_bookings["revenue_inr"].sum())
print("  scored rows (2025-26 only): %d" % fact_bookings["cancellation_risk"].notna().sum())

fact_bookings: 7171 rows x 31 cols
  revenue check: Rs 54950350
  scored rows (2025-26 only): 2396


## `fact_trips`

Trip grain - the capacity denominator. `capacity_utilization` in the dashboard is
`SUM(fact_trips.completed_seats) / SUM(fact_trips.boat_capacity)`.

In [4]:
seats = bk.groupby("trip_id").agg(
    seats_sold=("booking_id", "size"),
    completed_seats=("booking_id", lambda s: 0),
)
seats["completed_seats"] = (bk[bk.status == "Completed"].groupby("trip_id").size()
                            .reindex(seats.index, fill_value=0))
seats["cancelled_seats"] = (bk[bk.status == "Cancelled"].groupby("trip_id").size()
                            .reindex(seats.index, fill_value=0))
seats["no_show_seats"] = (bk[bk.status == "No-show"].groupby("trip_id").size()
                          .reindex(seats.index, fill_value=0))
trip_date = bk.groupby("trip_id").agg(dive_date=("dive_date", "first"),
                                      season_year=("season_year", "first"),
                                      dive_month=("dive_month", "first"),
                                      dive_ym=("dive_ym", "first"))

fact_trips = (tp.set_index("trip_id").join(seats).join(trip_date).reset_index())
fact_trips["season"] = np.select(
    [fact_trips["dive_month"].isin([12, 1, 2, 3]),
     fact_trips["dive_month"].isin([4, 5, 6, 9, 10, 11])],
    ["Peak", "Shoulder"], default="Monsoon-Closed")   # A-OQ4, same as dim_date
fact_trips["empty_seats"] = fact_trips["boat_capacity"] - fact_trips["completed_seats"]
fact_trips["never_sold_seats"] = fact_trips["boat_capacity"] - fact_trips["seats_sold"]
fact_trips["dive_date"] = fact_trips["dive_date"].dt.strftime("%Y-%m-%d")
print("fact_trips: %d rows x %d cols" % fact_trips.shape)
print("  total slots %d | completed seats %d | utilization %.1f%%"
      % (fact_trips.boat_capacity.sum(), fact_trips.completed_seats.sum(),
         100 * fact_trips.completed_seats.sum() / fact_trips.boat_capacity.sum()))

fact_trips: 922 rows x 15 cols
  total slots 9230 | completed seats 5493 | utilization 59.5%


## `fact_inquiries` and `dim_customers`

In [5]:
fact_inquiries = iq.copy()
# booking_conversion_rate counts an inquiry only if its booking COMPLETED (A-OQ1)
completed_ids = set(bk.loc[bk.status == "Completed", "booking_id"])
fact_inquiries["is_converted"] = (fact_inquiries["converted_booking_id"]
                                  .isin(completed_ids).astype(int))
fact_inquiries["is_converted_any_booking"] = fact_inquiries["converted"].astype(int)
fact_inquiries["inquiry_date"] = fact_inquiries["inquiry_date"].dt.strftime("%Y-%m-%d")
fact_inquiries = fact_inquiries[["inquiry_id", "inquiry_date", "channel",
                                 "converted_booking_id", "is_converted",
                                 "is_converted_any_booking", "season_year"]]

comp = bk[bk.status == "Completed"]
dim_customers = cu.copy()
dim_customers["completed_bookings"] = (comp.groupby("customer_id").size()
                                       .reindex(cu["customer_id"], fill_value=0).values)
dim_customers["lifetime_revenue_inr"] = (comp.groupby("customer_id")["revenue_inr"].sum()
                                         .reindex(cu["customer_id"]).values)
dim_customers["is_repeat_customer"] = (dim_customers["completed_bookings"] > 1).astype(int)
took_course = set(comp.loc[comp.dive_type == "Course", "customer_id"])
dim_customers["took_course"] = dim_customers["customer_id"].isin(took_course).astype(int)

first_course = comp[comp.dive_type == "Course"].groupby("customer_id")["dive_date"].min()
fun = comp[comp.dive_type == "Fun Dive"][["customer_id", "dive_date"]]
returned = set(fun.merge(first_course.rename("fc"), on="customer_id")
                  .query("dive_date > fc")["customer_id"])
dim_customers["course_then_fundive"] = dim_customers["customer_id"].isin(returned).astype(int)
dim_customers["signup_date"] = dim_customers["signup_date"].dt.strftime("%Y-%m-%d")

print("fact_inquiries: %d rows | dim_customers: %d rows" % (len(fact_inquiries),
                                                            len(dim_customers)))
print("  repeat customers %d | course customers %d | course->fun %d"
      % (dim_customers.is_repeat_customer.sum(), len(first_course),
         dim_customers.course_then_fundive.sum()))

fact_inquiries: 10845 rows | dim_customers: 5495 rows
  repeat customers 877 | course customers 860 | course->fun 185


## `kpi_summary` - the card source

Pre-computed because several section-3 KPIs cannot be derived from
`fact_bookings` alone (`booking_conversion_rate` needs inquiries,
`repeat_customer_rate` and `customer_ltv` need customer grain). Cards read this
table directly so the dashboard cannot disagree with the notebook.

In [6]:
kpi_out = kpis.copy()
kpi_out["display_value"] = np.where(
    kpi_out["unit"] == "ratio", (100 * kpi_out["value"]).round(1).astype(str) + "%",
    np.where(kpi_out["unit"] == "INR", "Rs " + kpi_out["value"].round(0).astype(int).astype(str),
             kpi_out["value"].round(1).astype(str)))
kpi_out.to_csv(os.path.join(DASH, "kpi_summary.csv"), index=False)

SECTION3 = ["capacity_utilization", "booking_conversion_rate", "cancellation_rate",
            "no_show_rate", "repeat_customer_rate", "course_to_fundive_conversion",
            "revenue_per_customer", "revenue_per_dive", "customer_ltv",
            "seasonality_index"]
present = set(kpi_out["kpi"])
missing = [k for k in SECTION3 if k not in present]
print("section-3 KPIs present in export: %d of %d" % (len(SECTION3) - len(missing),
                                                      len(SECTION3)))
assert not missing, "MISSING section-3 KPIs: %s" % missing
print(kpi_out[kpi_out.scope == "overall"][["kpi", "display_value"]].to_string(index=False))

section-3 KPIs present in export: 10 of 10
                                 kpi display_value
                capacity_utilization         59.5%
             booking_conversion_rate         39.3%
booking_conversion_rate_all_bookings         51.3%
                   cancellation_rate         18.6%
                        no_show_rate          4.8%
                repeat_customer_rate         16.0%
        course_to_fundive_conversion         21.5%
                revenue_per_customer      Rs 10000
                    revenue_per_dive       Rs 4450
                        customer_ltv      Rs 10000
                       total_revenue   Rs 54950350
               mean_revenue_per_seat      Rs 10035
           non_weather_cancellations         795.0
      non_weather_cancellation_value    Rs 7977635


## Utilization heatmap (pre-shaped)

Long form, one row per season_year x month, ready to drop into a matrix visual.

In [7]:
heat = (fact_trips.groupby(["season_year", "dive_month"])
        .agg(boat_slots=("boat_capacity", "sum"),
             completed_seats=("completed_seats", "sum"),
             trips=("trip_id", "size")).reset_index())
heat["capacity_utilization"] = heat["completed_seats"] / heat["boat_slots"]
heat["month_name"] = heat["dive_month"].map(lambda m: MONTHS[m - 1])
heat.to_csv(os.path.join(DASH, "utilization_heatmap.csv"), index=False)
print(heat.pivot_table(index="season_year", columns="month_name",
                       values="capacity_utilization").mul(100).round(1).to_string())

month_name       Apr   Dec   Feb   Jan   Jun   Mar   May   Nov   Oct   Sep
season_year                                                               
2023-24 (half)  67.0   NaN  69.2  70.7   NaN  68.3  63.3   NaN   NaN   NaN
2024-25         55.1  59.0  63.6  61.9  10.3  61.7  53.1  59.0  55.3   9.8
2025-26         49.5  61.1  58.3  61.2   7.2  61.6  50.6  52.2  55.1  13.0


In [8]:
for name, df in [("fact_bookings", fact_bookings), ("fact_trips", fact_trips),
                 ("fact_inquiries", fact_inquiries), ("dim_customers", dim_customers),
                 ("dim_date", dim_date)]:
    p = os.path.join(DASH, name + ".csv")
    df.to_csv(p, index=False)
    print("%-16s %6d rows x %2d cols -> dashboard/%s.csv" % (name, len(df), df.shape[1], name))

print("\nreconciliation against the notebook KPIs:")
print("  capacity_utilization  %.3f" % (fact_trips.completed_seats.sum()
                                        / fact_trips.boat_capacity.sum()))
print("  cancellation_rate     %.3f" % fact_bookings.is_cancelled.mean())
print("  no_show_rate          %.3f" % fact_bookings.is_no_show.mean())
print("  booking_conversion    %.3f" % fact_inquiries.is_converted.mean())
print("  repeat_customer_rate  %.3f" % dim_customers.is_repeat_customer.mean())
print("  revenue_per_dive      %.0f" % (fact_bookings.revenue_inr.sum()
                                        / fact_bookings.completed_dives.sum()))
print("  total_revenue         %.0f" % fact_bookings.revenue_inr.sum())

fact_bookings      7171 rows x 31 cols -> dashboard/fact_bookings.csv
fact_trips          922 rows x 15 cols -> dashboard/fact_trips.csv


fact_inquiries    10845 rows x  7 cols -> dashboard/fact_inquiries.csv
dim_customers      5495 rows x 14 cols -> dashboard/dim_customers.csv
dim_date            882 rows x 11 cols -> dashboard/dim_date.csv

reconciliation against the notebook KPIs:
  capacity_utilization  0.595
  cancellation_rate     0.186
  no_show_rate          0.048
  booking_conversion    0.393
  repeat_customer_rate  0.160
  revenue_per_dive      4450
  total_revenue         54950350
